## 1. Database design

```
Database (ndta631.db)
│
├── ict_regulatory
│
└── cybersecurity_index
```

Both tables share the same shape — one row per (country, indicator, year) — so the two datasets can be queried and joined consistently.

In [1]:
import sqlite3
import pandas as pd

DB_PATH = "../database/ndta631.db"
conn = sqlite3.connect(DB_PATH)
print("Connected to", DB_PATH)

Connected to ../database/ndta631.db


## 2. CREATE — inspect the schema

In [2]:
cur = conn.cursor()
cur.execute("SELECT name, sql FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';")
for name, sql in cur.fetchall():
    print(f"--- {name} ---")
    print(sql)
    print()

--- ict_regulatory ---
CREATE TABLE ict_regulatory (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    country_code    TEXT NOT NULL,
    country_name    TEXT NOT NULL,
    indicator_code  TEXT NOT NULL,
    indicator_name  TEXT NOT NULL,
    unit            TEXT,
    year            INTEGER NOT NULL,
    value           REAL,
    UNIQUE(country_code, indicator_code, year)
)

--- cybersecurity_index ---
CREATE TABLE cybersecurity_index (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    country_code    TEXT NOT NULL,
    country_name    TEXT NOT NULL,
    indicator_code  TEXT NOT NULL,
    indicator_name  TEXT NOT NULL,
    unit            TEXT,
    year            INTEGER NOT NULL,
    value           REAL,
    UNIQUE(country_code, indicator_code, year)
)



## 3. INSERT — confirm row counts

In [3]:
for table in ["ict_regulatory", "cybersecurity_index"]:
    cur.execute(f"SELECT COUNT(*) FROM {table};")
    print(f"{table}: {cur.fetchone()[0]} rows")

ict_regulatory: 85 rows
cybersecurity_index: 51 rows


## 4. SELECT — query the data

In [4]:
query = """
SELECT year, value
FROM ict_regulatory
WHERE indicator_code = 'ITU_ICT_REG_OVRL_SCR'
ORDER BY year;
"""
pd.read_sql_query(query, conn)

,year,value
0,2007,41.3333
1,2008,43.3333
2,2009,43.3333
3,2010,52.3333
4,2011,52.3333
5,2012,52.3333
6,2013,54.3333
7,2014,61.3333
8,2015,66.6667
9,2016,66.6667


In [5]:
query = """
SELECT year, value
FROM cybersecurity_index
WHERE indicator_code = 'ITU_GCI_GCI_OVRL_SCRE'
ORDER BY year;
"""
pd.read_sql_query(query, conn)

,year,value
0,2020,78.4629
1,2024,86.2463


### JOIN — regulatory score vs. cybersecurity score, common year (2020)

In [6]:
query = """
SELECT r.year,
       r.value AS regulatory_overall_score,
       c.value AS cybersecurity_overall_score
FROM ict_regulatory r
JOIN cybersecurity_index c
  ON r.year = c.year AND r.country_code = c.country_code
WHERE r.indicator_code = 'ITU_ICT_REG_OVRL_SCR'
  AND c.indicator_code = 'ITU_GCI_GCI_OVRL_SCRE';
"""
pd.read_sql_query(query, conn)

,year,regulatory_overall_score,cybersecurity_overall_score
0,2020,86.0,78.4629


## 5. UPDATE — safely update a single record

Updates are always scoped to one row via its primary key (`id`), never a broad `WHERE` clause,
so no other data is put at risk.

In [7]:
cur.execute("""
    SELECT id, value FROM ict_regulatory
    WHERE indicator_code = 'ITU_ICT_REG_AUTH' AND year = 2022;
""")
row_id, old_value = cur.fetchone()
print("Before:", row_id, old_value)

cur.execute("UPDATE ict_regulatory SET value = 18.5 WHERE id = ?;", (row_id,))
conn.commit()

cur.execute("SELECT id, value FROM ict_regulatory WHERE id = ?;", (row_id,))
print("After: ", cur.fetchone())

# Revert, so the notebook can be re-run without permanently altering the data
cur.execute("UPDATE ict_regulatory SET value = ? WHERE id = ?;", (old_value, row_id))
conn.commit()
print("Reverted back to:", old_value)

Before: 25 18.0
After:  (25, 18.5)
Reverted back to: 18.0


## 6. DELETE — safely delete a single record

A temporary test row is inserted first, then deleted, so the DELETE demo never touches real data.

In [8]:
cur.execute("""
    INSERT INTO cybersecurity_index
        (country_code, country_name, indicator_code, indicator_name, unit, year, value)
    VALUES ('ZAF', 'South Africa', 'TEST_ROW_FOR_DELETE_DEMO', 'Temporary test row', NULL, 1999, 0.0);
""")
conn.commit()

cur.execute("SELECT id FROM cybersecurity_index WHERE indicator_code = 'TEST_ROW_FOR_DELETE_DEMO';")
test_id = cur.fetchone()[0]
print("Inserted test row id:", test_id)

cur.execute("DELETE FROM cybersecurity_index WHERE id = ?;", (test_id,))
conn.commit()

cur.execute("SELECT * FROM cybersecurity_index WHERE id = ?;", (test_id,))
print("Row after delete (should be None):", cur.fetchone())

Inserted test row id: 53
Row after delete (should be None): None


## 7. Pandas integration — load the whole database back into DataFrames

In [9]:
ict_df = pd.read_sql_query("SELECT * FROM ict_regulatory ORDER BY year;", conn)
gci_df = pd.read_sql_query("SELECT * FROM cybersecurity_index ORDER BY year;", conn)

print("ict_regulatory:", ict_df.shape)
print("cybersecurity_index:", gci_df.shape)
ict_df.head()

ict_regulatory: (85, 8)
cybersecurity_index: (51, 8)


,id,country_code,country_name,indicator_code,indicator_name,unit,year,value
0,11,ZAF,South Africa,ITU_ICT_REG_AUTH,Regulatory Authority Score (ITU Regulatory Tra...,0_TO_100,2007,12.0000
1,26,ZAF,South Africa,ITU_ICT_REG_CMP_FRMWK,Competition Framework Score (ITU Regulatory Tr...,0_TO_100,2007,10.3333
2,41,ZAF,South Africa,ITU_ICT_REG_MNDTE,Regulatory Mandate Score (ITU Regulatory Tracker),0_TO_100,2007,17.0000
3,56,ZAF,South Africa,ITU_ICT_REG_OVRL_SCR,ICT Regulatory Tracker - Overall Score (ITU Re...,0_TO_100,2007,41.3333
4,71,ZAF,South Africa,ITU_ICT_REG_RGME,Regulatory Regime Score (ITU Regulatory Tracker),0_TO_100,2007,2.0000


In [10]:
gci_df.head()

,id,country_code,country_name,indicator_code,indicator_name,unit,year,value
0,1,ZAF,South Africa,ITU_GCI_CDS_CYB_PROF_TRAIN,Training for Cybersecurity Professionals - CDS...,0_TO_1,2020,0.7706
1,3,ZAF,South Africa,ITU_GCI_CDS_GOV_ACAD,Educational programs or academic curricula in ...,0_TO_1,2020,1.0000
2,5,ZAF,South Africa,ITU_GCI_CDS_GOV_MCNSM,Government Incentive Mechanisms - CDS6 (ITU GCI),0_TO_1,2020,0.0000
3,7,ZAF,South Africa,ITU_GCI_CDS_NAT_CYB_IND,National cybersecurity industry - CDS5 (ITU GCI),0_TO_1,2020,1.0000
4,9,ZAF,South Africa,ITU_GCI_CDS_PUB_CYB_AWE,Public cybersecurity awareness campaigns - CDS...,0_TO_1,2020,0.8751


In [11]:
conn.close()
print("Connection closed.")

Connection closed.
